# Semantic map of sources considered in the PETs survey

This notebook produces the figure inserted between the Introduction and Section 2 of the survey. It walks through the seven pipeline stages documented in `docs/methodology.md`:

1. Parse the bibliography into structured metadata (`src/parse_references.py`)
2. Acquire text appropriate to each source type (`src/acquire_text.py`)
3. Normalise text into a comparable form (`src/preprocess.py`)
4. Compute TF-IDF and optional sentence embeddings (`src/features.py`)
5. Assign topics against a controlled vocabulary (`src/topic_assignment.py`)
6. Build a typed multi-edge graph (`src/graph.py`)
7. Render the figure (`src/visualise.py`)

Every stage runs even if earlier stages have not been completed end-to-end: where text acquisition has not yet been performed, the notebook falls back to the title-plus-venue payload that the parser produced. The figure quality improves monotonically as more abstracts are added to `data/abstracts_cache/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import topic_assignment as ta
from src import features as feat
from src import graph as graph_mod
from src import visualise as viz

DATA = ROOT / 'data'
FIGURES = ROOT / 'figures'
VOCAB = DATA / 'topic_vocabulary.yaml'
SOURCES = DATA / 'sources.csv'
SOURCE_TEXT = DATA / 'source_text.csv'

pd.set_option('display.max_colwidth', 80)

## Stage 1 - structured metadata

`sources.csv` is the audited source of truth. It is produced by `python -m src.parse_references`, which is deterministic given the bibliography. Each row carries:

- mechanical fields (`id`, `number`, `year`, `authors`, `title`, `url`, `doi`, `arxiv_id`)
- heuristic seeds (`publication_type_seed`, `pet_family_seed`, `text_strategy`)
- curation fields (`publication_type`, `pet_family`, `primary_topic`, `secondary_topics`, `review_note`)

Seeds are starting points only. The curation workflow in `docs/topic_assignment_guide.md` describes how to lift them into authoritative labels.

In [ ]:
sources = pd.read_csv(SOURCES)
print(f'rows: {len(sources)}')
print(f'years missing: {sources.year.isna().sum()}')
print(f'urls present: {(sources.url.astype(str).str.len() > 0).sum()}')
print('\nseed publication type distribution:')
print(sources.publication_type_seed.value_counts(dropna=False))

## Stage 2 - text payload

If `data/source_text.csv` exists (produced by `python -m src.acquire_text`) the curated text payload is used. Otherwise we fall back to title + venue, which is enough to produce a plausible draft figure but not the final one. The fallback is flagged in the cell output so it is impossible to mistake a draft for the published figure.

In [ ]:
if SOURCE_TEXT.exists():
    text_df = pd.read_csv(SOURCE_TEXT).fillna({'text': ''})
    sources_text = sources.merge(text_df[['id', 'text']], on='id', how='left')
    using_fallback = sources_text.text.isna() | (sources_text.text == '')
    if using_fallback.any():
        fallback = (sources_text.title.fillna('') + ' ' + sources_text.venue.fillna('')).str.strip()
        sources_text.loc[using_fallback, 'text'] = fallback[using_fallback]
    print(f'using acquired text for {(~using_fallback).sum()} sources, fallback for {using_fallback.sum()}')
else:
    sources_text = sources.copy()
    sources_text['text'] = (sources_text.title.fillna('') + ' ' + sources_text.venue.fillna('')).str.strip()
    print('source_text.csv not found -> using title+venue fallback for ALL sources.')
    print('This is a draft figure. Run src/acquire_text.py to produce the published one.')

## Stages 3-4 - normalisation and features

We normalise text and compute TF-IDF. Sentence embeddings are optional; the rest of the pipeline runs without them.

In [ ]:
try:
    from src import preprocess
    sources_text['text_normalised'] = preprocess.normalise_series(sources_text.text.tolist(), VOCAB)
    normalised_ok = True
except Exception as e:
    print(f'preprocess unavailable ({e!s}); using raw text')
    sources_text['text_normalised'] = sources_text.text.str.lower()
    normalised_ok = False

features = feat.compute_tfidf(
    sources_text.id.tolist(),
    sources_text.text_normalised.tolist(),
    min_df=1,
)
print(f'TF-IDF vocabulary size: {len(features.tfidf_vocab)} (normalised: {normalised_ok})')

## Stage 5 - topic assignment

We score each source against the controlled vocabulary and write the suggestions to `data/topic_suggestions.csv`. If `primary_topic` is already populated in `sources.csv` (human curation has taken place) the suggestion is shown alongside, never overriding the human decision.

In [ ]:
suggestions = ta.score_topics(
    sources_text.id.tolist(),
    sources_text.text_normalised.tolist(),
    VOCAB,
)
ta.write_suggestions(suggestions, DATA / 'topic_suggestions.csv')

suggestion_df = pd.DataFrame([{'id': s.source_id,
                                'suggested_topic': s.primary_topic,
                                'margin': round(s.margin, 3),
                                'confident': s.is_confident()} for s in suggestions])
merged = sources_text.merge(suggestion_df, on='id', how='left')
merged['primary_topic'] = merged.primary_topic.where(merged.primary_topic.astype(bool), merged.suggested_topic)
merged[['id','title','primary_topic','suggested_topic','margin','confident']].head(10)

## Stage 6 - graph construction

Three edge types are produced and stored on a `MultiGraph`:

- `topic` edges link every pair of sources that share the same primary topic;
- `family` edges link every pair sharing at least one PET family tag;
- `semantic` edges are k-nearest neighbours on TF-IDF cosine similarity above a threshold.

Each edge carries an `etype` attribute. The next stage filters on it.

In [ ]:
sim = feat.pairwise_cosine(features.tfidf_matrix)
g = graph_mod.build_graph(merged, similarity=sim, k_semantic=5, semantic_threshold=0.20)
print(f'nodes: {g.number_of_nodes()}')
from collections import Counter
edge_breakdown = Counter(d['etype'] for _, _, d in g.edges(data=True))
print('edges by type:', dict(edge_breakdown))

## Stage 7 - render

The main figure shows topic and semantic edges. Node colour is the primary topic; node shape is the source type; node size is degree. The colour palette is a colour-blind-safe qualitative scheme.

In [ ]:
FIGURES.mkdir(exist_ok=True)
fig = viz.render_graph(
    g,
    out_path=FIGURES / 'semantic_map_main.png',
    show_edges=('topic', 'semantic'),
)
fig

In [ ]:
viz.render_supporting(merged, FIGURES)
print('Supporting figures written to', FIGURES)
print()
print('Three supporting figures:')
print('  - timeline.png              sources by year and primary topic')
print('  - type_topic_heatmap.png    source type x primary topic crosstab')
print('  - pet_family_breakdown.png  counts across the seven survey PET families,')
print('                              stacked by primary topic. Sources that carry no')
print('                              tag from the seven appear as "Other / none".')


## Reflections and limitations

The figure depicts the sources considered in this survey, not the entire PETs literature. Several methodological choices shape what it can and cannot say:

- **Text-length control**: long technical white papers contribute only their executive summary, while short blog posts contribute their full body. This avoids letting page count dominate similarity but means that a single deeply technical paper and a short policy summary are treated as roughly equivalent inputs.
- **Closed topic set**: topics are defined to mirror the structure of the survey rather than discovered from the data. A reader looking at the figure is reading our taxonomy of the literature, not a neutral one.
- **Heuristic seeds**: publication type and PET family begin as keyword matches. Where these have not yet been confirmed by a reviewer, the relevant column carries the seed value with `_seed` provenance. Any cluster that surprises the reader should be cross-checked against the seed audit log.

Any visible imbalance (e.g. an over-representation of `governance` or DP-focused sources) is treated as an analytical observation about the evidence base, in the spirit of Page et al. (2021)'s PRISMA 2020 guidance on transparent reporting of inclusion patterns.